In [1]:
system_prompt = """
You are a creative writer for fantasy world-building.
Instructions:
- Never ask clarifying questions.
- Never request more information.
- Immediately generate original creative content based on the request.
- Only generate in plain text.
- Stay below 3-5 sentences for the description.
- Always follow the exact output format requested.
"""



In [2]:

world_prompt = """
Generate a creative description for a unique fantasy world with an
interesting concept where French is used as the only way to do magic spell. The world has castles, schools, and small villages.
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

output = client.chat.completions.create(
    model="qwen2.5:14b",
    temperature=0.8,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": world_prompt}
    ],
)
Example output format:
World Name: Verdanth
World Description: A mist-covered realm where floating islands drift...

Now generate a NEW original world in the same format:
World Name:
World Description:
"""

In [3]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

output = client.chat.completions.create(
    model="qwen2.5:14b",
    temperature=0.8,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": world_prompt}
    ],
)

In [4]:
world_output =output.choices[0].message.content
print(world_output)

World Name: Lunevraie
World Description: A world where the ethereal glow of moonlight intertwines with the arcane, and the ancient language of French is the key to wielding magic. Castles tower over sprawling enchanted gardens, schools echo with the chants of young sorcerers learning the mystical art of spellcasting, and villages hum with the whispers of powerful incantations.


In [5]:
world_output = world_output.strip()
world = {
    "name": world_output.split('\n')[0].strip()
    .replace('World Name: ', ''),
    "description": '\n'.join(world_output.split('\n')[1:])
    .replace('World Description:', '').strip()
}

In [8]:
kingdom_prompt = f"""
Create 3 different kingdoms for a fantasy world.
For each kingdom generate a description based on the world it's in. \
Describe important leaders, cultures, history of the kingdom.\

Output content in the form:
Kingdom 1 Name: <KINGDOM NAME>
Kingdom 1 Description: <KINGDOM DESCRIPTION>
Kingdom 2 Name: <KINGDOM NAME>
Kingdom 2 Description: <KINGDOM DESCRIPTION>
Kingdom 3 Name: <KINGDOM NAME>
Kingdom 3 Description: <KINGDOM DESCRIPTION>

World Name: {world['name']}
World Description: {world['description']}

Kingdom 1"""

In [10]:


output = client.chat.completions.create(
    model="qwen2.5:14b",
    temperature=0.8,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": kingdom_prompt}
    ]
)

In [11]:
kingdoms = {}
kingdoms_output = output.choices[0].message.content

for output in kingdoms_output.split('\n\n'):
  kingdom_name = output.strip().split('\n')[0] \
    .split('Name: ')[1].strip()
  print(f'Created kingdom "{kingdom_name}" in {world["name"]}')
  kingdom_description = output.strip().split('\n')[1] \
    .split('Description: ')[1].strip()
  kingdom = {
      "name": kingdom_name,
      "description": kingdom_description,
      "world": world['name']
  }
  kingdoms[kingdom_name] = kingdom
world['kingdoms'] = kingdoms

print(f'\nKingdom 1 Description: \
{kingdom["description"]}')

Created kingdom "Éclairia" in Lunevraie
Created kingdom "Verseau" in Lunevraie
Created kingdom "Crépuscule" in Lunevraie

Kingdom 1 Description: Crépuscule is led by Archduke Gabriel, a ruler who wields the dark magic of twilight, blending the power of dusk and dawn. The kingdom is cloaked in perpetual twilight, where night and day merge, allowing for unique magical phenomena. Crépuscule’s culture emphasizes secrecy and mystery, with a strong tradition of shadow magic and espionage. Its history is filled with tales of the elusive Shadow Council, a secretive group that guides the kingdom’s destiny with arcane knowledge and hidden agendas.


In [12]:
def get_town_prompt(world, kingdom):
    return f"""
    Create 3 different towns for a fantasy kingdom abd world. \
    Describe the region it's in, important places of the town, \
    and interesting history about it. \
    
    Output content in the form:
    Town 1 Name: <TOWN NAME>
    Town 1 Description: <TOWN DESCRIPTION>
    Town 2 Name: <TOWN NAME>
    Town 2 Description: <TOWN DESCRIPTION>
    Town 3 Name: <TOWN NAME>
    Town 3 Description: <TOWN DESCRIPTION>
    
    World Name: {world['name']}
    World Description: {world['description']}
    
    Kingdom Name: {kingdom['name']}
    Kingdom Description {kingdom['description']}
    
    Town 1 Name:"""


In [13]:
def create_towns(world, kingdom):
    print(f'\nCreating towns for kingdom: {kingdom["name"]}...')
    output = client.chat.completions.create(
  model="qwen2.5:14b",
    temperature=0.8,
        messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": get_town_prompt(world, kingdom)}
      ],
  )
    towns_output = output.choices[0].message.content
    
    towns = {}
    for output in towns_output.split('\n\n'):
        town_name = output.strip().split('\n')[0]\
        .split('Name: ')[1].strip()
        print(f'- {town_name} created')
        
        town_description = output.strip().split('\n')[1]\
        .split('Description: ')[1].strip()
        
        town = {
          "name": town_name,
          "description": town_description,
          "world": world['name'],
          "kingdom": kingdom['name']
        }
        towns[town_name] = town
    kingdom["towns"] = towns

In [14]:
for kingdom in kingdoms.values():
    create_towns(world, kingdom)  

town = list(kingdom['towns'].values())[0]
print(f'\nTown 1 Description: \
{town["description"]}')



Creating towns for kingdom: Éclairia...
- Luciere created
- Cendres Étoilées created
- Lune Clair created

Creating towns for kingdom: Verseau...
- Éclaire created
- Lunevraie-Ville created
- Port Eauveille created

Creating towns for kingdom: Crépuscule...
- Éclat d'Argent created
- Sombres Ardents created
- Verge d'Émeraude created

Town 1 Description: Éclat d'Argent is nestled in a valley of silvery moonlight, where the air is thick with whispers of ancient spells and the glow of arcane runes adorns every stone. The town's central market is a bustling hub of enchanting commerce, while the nearby Académie des Étoiles trains promising young mages in the art of celestial magic. Legend has it that Éclat d'Argent was built on the site of an ancient moon temple, and its moon-aligned architecture grants its residents enhanced magical abilities under the full moon.


In [25]:
def get_npc_prompt(world, kingdom, town): 
    return f"""
    Create 3 different characters based on the world, kingdom \
    and town they're in. Describe the character's appearance and \
    profession, as well as their deeper pains and desires.  \
    
    Output content in the form:
    Character 1 Name: <CHARACTER NAME>
    Character 1 Description: <CHARACTER DESCRIPTION>
    Character 2 Name: <CHARACTER NAME>
    Character 2 Description: <CHARACTER DESCRIPTION>
    Character 3 Name: <CHARACTER NAME>
    Character 3 Description: <CHARACTER DESCRIPTION>
    
    World Name: {world['name']}
    World Description: {world['description']}
    
    Kingdom Name: {kingdom['name']}
    Kingdom Description: {kingdom['description']}
    
    Town Name: {town['name']}
    Town Description: {town['description']}
    
    Character 1 Name:"""

In [28]:
def create_npcs(world, kingdom, town):
    print(f'\nCreating characters for the town of: {town["name"]}...')
    output = client.chat.completions.create(
  model="qwen2.5:14b",
    temperature=1,        
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_npc_prompt(world, kingdom, town)}
        ],
    )

    npcs_output = output.choices[0].message.content
    npcs = {}
    for output in npcs_output.split('\n\n'):
        npc_name = output.strip().split('\n')[0]\
        .split('Name: ')[1].strip()
        print(f'- "{npc_name}" created')
        
        npc_description = output.strip().split('\n')[1\
        ].split('Description: ')[1].strip()
        
        npc = {
        "name": npc_name,
        "description": npc_description,
        "world": world['name'],
        "kingdom": kingdom['name'],
        "town": town['name']
        }
        npcs[npc_name] = npc
    town["npcs"] = npcs

In [29]:
for kingdom in kingdoms.values():
    for town in kingdom['towns'].values():
        create_npcs(world, kingdom, town)
  # For now we'll only generate npcs for one kingdom
    break


Creating characters for the town of: Luciere...
- "Alaric Éclair" created
- "Sylvie Vespera" created
- "Lucien Thalass" created

Creating characters for the town of: Cendres Étoilées...
- "Léona" created
- "Étienne" created
- "Mélisande" created

Creating characters for the town of: Lune Clair...
- "Alain de Lune" created
- "Elise d'Éclair" created
- "Mathieu Leclair" created


In [30]:
import json

def save_world(world, filename):
    with open(filename, 'w') as f:
        json.dump(world, f)

def load_world(filename):
    with open(filename, 'r') as f:
        return json.load(f)

#save_world(world, '../shared_data/Kyropeia.json')
save_world(world, 'Lunevraie_L1.json')